# AI Learning Environment Setup

一键准备 Linux + NVIDIA GPU 的 AI 学习环境。

执行 `Run All` 后将依次完成：
- 检查系统与 GPU
- 安装系统依赖
- 安装 Miniconda
- 创建 `llm-learning` / Python 3.10 环境
- 安装 CUDA 12.8 (cu128) GPU 版 PyTorch
- 安装常用 AI 依赖
- 注册 Jupyter Kernel
- 验证 PyTorch、CUDA 和 GPU


In [ ]:
import os
import platform
import subprocess

CONDA_DIR = os.path.expanduser("~/miniconda3")
ENV_NAME = "llm-learning"
PYTHON_VERSION = "3.10"

print("OS:", platform.system())
print("Architecture:", platform.machine())
print("Conda:", CONDA_DIR)
print("Environment:", ENV_NAME)
print("Python:", PYTHON_VERSION)

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print("NVIDIA GPU detected:")
    subprocess.run(["nvidia-smi"])
else:
    print("WARNING: nvidia-smi not found.")
    print("Current environment may not have an NVIDIA GPU.")

In [ ]:
!sudo apt update
!sudo apt install -y wget bzip2

In [ ]:
import platform
import os

arch = platform.machine()

if arch == "x86_64":
    installer = "Miniconda3-latest-Linux-x86_64.sh"
elif arch == "aarch64":
    installer = "Miniconda3-latest-Linux-aarch64.sh"
else:
    raise RuntimeError(f"Unsupported architecture: {arch}")

installer_path = os.path.abspath(installer)

if os.path.exists(installer_path):
    print("Installer already exists:", installer_path)
else:
    url = f"https://repo.anaconda.com/miniconda/{installer}"
    print("Downloading:", url)
    !wget -q {url} -O {installer_path}

print("Installer:", installer_path)

In [ ]:
import os

if os.path.exists(CONDA_DIR):
    print("Miniconda already installed:", CONDA_DIR)
else:
    !bash {installer_path} -b -p {CONDA_DIR}
    print("Miniconda installation completed.")

In [ ]:
import os
import subprocess

print("Anaconda Terms of Service accepted.")

conda = os.path.expanduser("~/miniconda3/bin/conda")
ENV_NAME = "llm-learning"
PYTHON_VERSION = "3.10"

subprocess.run([
    conda,
    "tos", "accept",
    "--override-channels",
    "--channel",
    "https://repo.anaconda.com/pkgs/main"
], check=True)

subprocess.run([
    conda,
    "tos", "accept",
    "--override-channels",
    "--channel",
    "https://repo.anaconda.com/pkgs/r"
], check=True)

result = subprocess.run(
    [
        conda,
        "create",
        "-y",
        "-n",
        ENV_NAME,
        f"python={PYTHON_VERSION}",
        "pip"
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

print("\n========== STDOUT ==========")
print(result.stdout)

print("\n========== STDERR ==========")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Conda environment creation failed.")

In [ ]:
env_python = os.path.join(CONDA_DIR, "envs", ENV_NAME, "bin", "python")

subprocess.run(
    [
        env_python, "-m", "pip", "install",
        "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/cu128"
    ],
    check=True
)

print("GPU PyTorch installation completed.")

In [ ]:
packages = [
    "numpy",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "jupyter",
    "ipykernel",
    "tqdm",
    "pillow",
    "ultralytics",
    "d2l"
]

subprocess.run(
    [env_python, "-m", "pip", "install", *packages],
    check=True
)

print("All Python dependencies installed.")

In [ ]:
subprocess.run(
    [
        env_python, "-m", "ipykernel", "install",
        "--user",
        "--name", ENV_NAME,
        "--display-name", f"Python ({ENV_NAME})"
    ],
    check=True
)

print(f"Jupyter kernel registered: Python ({ENV_NAME})")

In [ ]:
result = subprocess.run(
    [env_python, "--version"],
    capture_output=True,
    text=True
)

print(result.stdout)
print("Python executable:")
print(env_python)

In [ ]:
code = '''
import torch

print("PyTorch version:", torch.__version__)
print("Built with CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("WARNING: CUDA is not available.")
'''

subprocess.run([env_python, "-c", code], check=True)

In [ ]:
print("=" * 50)
print("AI Environment Setup Complete")
print("=" * 50)

print("Conda:")
subprocess.run([conda, "--version"])

print("\nPython:")
subprocess.run([env_python, "--version"])

print("\nPyTorch / CUDA:")
subprocess.run([
    env_python, "-c",
    "import torch; print('PyTorch:', torch.__version__); "
    "print('CUDA:', torch.version.cuda); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')"
])

print("\nJupyter Kernel:")
print(f"Python ({ENV_NAME})")